# SVD & Low-Rank Approximation

Companion notebook for the [SVD & Low-Rank Approximation](https://ml-viz.vercel.app/courses/linear-algebra/04-svd-and-low-rank) lesson on ML Viz.

We'll verify the rotate–stretch–rotate picture, check Eckart–Young numerically, and compress an image with truncated SVD.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("dark_background")
plt.rcParams["figure.facecolor"] = "#0f1117"
plt.rcParams["axes.facecolor"] = "#1a1d27"
plt.rcParams["axes.edgecolor"] = "#2e3347"
plt.rcParams["grid.color"] = "#2e3347"

rng = np.random.default_rng(42)

## 1. SVD = rotate · stretch · rotate

U and V are orthogonal (rotations/reflections); Σ is diagonal (axis-aligned stretch). Reassembling them recovers A exactly.

In [ ]:
A = rng.normal(size=(5, 3))
U, s, Vt = np.linalg.svd(A, full_matrices=False)

print("U orthogonal:", np.allclose(U.T @ U, np.eye(3)))
print("V orthogonal:", np.allclose(Vt @ Vt.T, np.eye(3)))
print("singular values (sorted, non-negative):", s.round(3))
print("reconstruction exact:", np.allclose(U @ np.diag(s) @ Vt, A))

## 2. The PCA connection

Right singular vectors of centered data = principal components; σ²/n = explained variances.

In [ ]:
X = rng.normal(size=(500, 2)) @ np.array([[2.0, 1.2], [0.0, 0.6]])
Xc = X - X.mean(axis=0)

# PCA via covariance eigendecomposition
evals, evecs = np.linalg.eigh(Xc.T @ Xc / len(Xc))
# PCA via SVD
U, s, Vt = np.linalg.svd(Xc, full_matrices=False)

print("eigen variances :", np.sort(evals)[::-1].round(4))
print("SVD variances   :", (s**2 / len(Xc)).round(4))

## 3. Eckart–Young, checked

The rank-k truncation error in Frobenius norm equals the root-sum-square of the discarded singular values — and no random rank-k matrix does better.

In [ ]:
A = rng.normal(size=(60, 40)) @ rng.normal(size=(40, 40))
U, s, Vt = np.linalg.svd(A, full_matrices=False)

k = 10
A_k = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
err = np.linalg.norm(A - A_k, "fro")
tail = np.sqrt((s[k:] ** 2).sum())
print(f"truncation error = {err:.4f}, discarded tail = {tail:.4f}")

# try to beat it with random rank-k matrices (spoiler: you can't)
best_random = min(
    np.linalg.norm(A - (rng.normal(size=(60, k)) @ np.linalg.lstsq(rng.normal(size=(60, k)), A, rcond=None)[0]), "fro")
    for _ in range(20)
)
print(f"best of 20 random rank-{k} attempts = {best_random:.4f}  (worse)")

## 4. Image compression

A synthetic 'image' compressed at increasing rank — watch structure return layer by layer.

In [ ]:
# build a structured image: gradients + blocks + a bit of noise
n = 128
yy, xx = np.mgrid[0:n, 0:n] / n
img = np.sin(6 * xx) + 0.8 * np.cos(4 * yy) + (xx > 0.5) * (yy > 0.5) * 1.5
img += 0.05 * rng.normal(size=img.shape)

U, s, Vt = np.linalg.svd(img, full_matrices=False)

fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for ax, k in zip(axes, [1, 3, 10, 128]):
    approx = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    kept = 100 * (s[:k]**2).sum() / (s**2).sum()
    ax.imshow(approx, cmap="magma")
    ax.set_title(f"rank {k} · {kept:.1f}% energy", fontsize=10)
    ax.axis("off")
plt.suptitle("Truncated SVD: the best rank-k views of the image")
plt.show()

plt.figure(figsize=(7, 3))
plt.semilogy(s, color="#6366f1", lw=2)
plt.xlabel("index")
plt.ylabel("singular value (log)")
plt.title("Spectrum: a few large values, then a noise floor")
plt.grid(alpha=0.4)
plt.show()

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Best rank-k approximation

The Eckart–Young theorem says the best rank-$k$ approximation of $A$ keeps the top $k$ singular triplets:

$$A_k = U_{:, :k} \, \Sigma_{:k, :k} \, V^\top_{:k, :}
\qquad\text{with}\qquad
\lVert A - A_k \rVert_F = \sqrt{\textstyle\sum_{i > k} \sigma_i^2}$$

Implement the truncation; the checks verify both the rank **and** that exact error formula.

In [ ]:
def rank_k_approx(A, k):
    """Best rank-k approximation of A (Eckart-Young), built from its SVD."""
    U, s, Vt = np.linalg.svd(np.asarray(A, dtype=float), full_matrices=False)

    # TODO(you): keep the first k columns of U, first k singular values,
    # first k rows of Vt, and multiply them back together
    # (hint: U[:, :k], np.diag(s[:k]), Vt[:k, :])
    return ...

In [ ]:
# Checks — run me
rng = np.random.default_rng(0)
A = rng.standard_normal((8, 6))

A2 = rank_k_approx(A, 2)
assert np.linalg.matrix_rank(A2) == 2, "the result must have rank k"

s = np.linalg.svd(A, compute_uv=False)
expected_err = np.sqrt(np.sum(s[2:] ** 2))
assert abs(np.linalg.norm(A - A2, 'fro') - expected_err) < 1e-9, \
    "Frobenius error must equal √(sum of discarded σ²) — Eckart-Young"

assert np.allclose(rank_k_approx(A, 6), A), "keeping every singular value reconstructs A exactly"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def rank_k_approx(A, k):
    U, s, Vt = np.linalg.svd(np.asarray(A, dtype=float), full_matrices=False)
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
```

</details>

### Exercise 2 — Compression ratio

Storing a full $m \times n$ matrix takes $mn$ numbers. The rank-$k$ version stores only the factors: $U_k$ ($m \cdot k$), the singular values ($k$), and $V_k^\top$ ($k \cdot n$). Compute how many times smaller that is — and notice the punchline of the last check: for $k$ too close to full rank, "compression" actually costs **more** than the original.

In [ ]:
def compression_ratio(m, n, k):
    """How many times smaller rank-k SVD storage is vs. the full m x n matrix."""
    # TODO(you): numbers needed for the full matrix
    full = ...

    # TODO(you): numbers needed for the rank-k factors: m*k + k + k*n
    compressed = ...

    return full / compressed

In [ ]:
# Checks — run me
assert abs(compression_ratio(100, 100, 10) - 10000 / 2010) < 1e-12, "100x100 at k=10 -> ~5x smaller"
assert abs(compression_ratio(1000, 500, 20) - 500000 / 30020) < 1e-12, "1000x500 at k=20 -> ~16.7x smaller"
assert compression_ratio(10, 10, 8) < 1, "k too close to full rank costs MORE than storing A"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def compression_ratio(m, n, k):
    full = m * n
    compressed = m * k + k + k * n
    return full / compressed
```

</details>

**Next:** [PCA](https://ml-viz.vercel.app/courses/pca-dimensionality/01-pca) puts this machinery to work on data, and the [course quiz](https://ml-viz.vercel.app/courses/linear-algebra/05-quiz) checks the whole course.